# 🤖 GoldMind - Model Training, Evaluation & Strategy Backtest
Complete machine learning pipeline for XAU/USD hourly forecasting:
1. Resample 1-minute data $\rightarrow$ 1-hour bars
2. Build 45+ stationary, scale-invariant features
3. **Time-series chronological split FIRST (Train 72%, Val 8%, Test 20%)**
4. **Feature selection SECOND (fitted strictly on Train set - NO Data Leakage)**
5. Train **Random Forest Regressor**, **XGBoost Regressor**, and **XGBoost Classifier**
6. Evaluate Regression & Directional metrics (MAE, RMSE, $R^2$, Directional Accuracy)
7. **Trading Simulation & Backtest (PnL, Sharpe Ratio, Max Drawdown, Bar vs Trade Win Rate, Profit Factor with spread costs)**
8. Export all metrics, predictions, and artifacts


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Ensure repository root is on Python path
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, roc_auc_score
from xgboost import XGBRegressor, XGBClassifier

from src.features import build_features, make_target, select_top_features

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CSV_PATH = "data/XAU_1m_data.csv"
OUT_DIR = "model_output"
os.makedirs(OUT_DIR, exist_ok=True)

N_FEATURES_SELECTED = 20
TARGET_HORIZON = 1
RANDOM_STATE = 42

TEST_SIZE = 0.20          # Last 20% of time as out-of-sample test
VAL_SIZE = 0.10           # 10% of train portion for XGB early stopping
SPREAD_PCT = 0.0002       # 0.02% (~$0.40 - $0.80 per gold trade) transaction cost / spread

print("Configuration initialized.")


In [ ]:
# ---------- 1. Load & Resample Data ----------
print(f"Loading {CSV_PATH} ...")
raw = (
    pd.read_csv(CSV_PATH, parse_dates=["Date"])
    .set_index("Date")
    .sort_index()
)
print(f"Loaded {len(raw):,} 1-minute bars ({raw.index.min()} to {raw.index.max()})")

df_1h = (
    raw.resample("1h")
    .agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"})
    .dropna()
)
print(f"Resampled to {len(df_1h):,} hourly bars.")


In [ ]:
# ---------- 2. Build Stationary Features & Targets ----------
print("Building stationary features ...")
feats = build_features(df_1h)
y_reg = make_target(df_1h, horizon=TARGET_HORIZON, kind="regression")
y_clf = make_target(df_1h, horizon=TARGET_HORIZON, kind="classification")

# Join and drop warmup NaNs
data = feats.join(y_reg.rename("target_reg")).join(y_clf.rename("target_clf")).dropna()
X_all = data.drop(columns=["target_reg", "target_clf"])
y_reg_all = data["target_reg"]
y_clf_all = data["target_clf"]
close_series = df_1h.loc[data.index, "Close"]

print(f"Clean samples: {len(data):,}  |  Features: {X_all.shape[1]}")


In [ ]:
# ---------- 3. Time-Series Split FIRST (No Data Leakage) ----------
n = len(X_all)
test_start = int(n * (1 - TEST_SIZE))
val_start = int(test_start * (1 - VAL_SIZE))

X_train, y_train_reg, y_train_clf = X_all.iloc[:val_start], y_reg_all.iloc[:val_start], y_clf_all.iloc[:val_start]
X_val, y_val_reg, y_val_clf = X_all.iloc[val_start:test_start], y_reg_all.iloc[val_start:test_start], y_clf_all.iloc[val_start:test_start]
X_test, y_test_reg, y_test_clf = X_all.iloc[test_start:], y_reg_all.iloc[test_start:], y_clf_all.iloc[test_start:]

test_close = close_series.iloc[test_start:]

print("Split sizes (Chronological):")
print(f"  Train : {len(X_train):,} bars ({X_train.index.min()} -> {X_train.index.max()})")
print(f"  Val   : {len(X_val):,} bars ({X_val.index.min()} -> {X_val.index.max()})")
print(f"  Test  : {len(X_test):,} bars ({X_test.index.min()} -> {X_test.index.max()})")


In [ ]:
# ---------- 4. Feature Selection on Train Set Only ----------
print(f"Selecting top {N_FEATURES_SELECTED} features using Train Set ONLY ...")
top_feats, train_importances = select_top_features(
    X_train, y_train_reg, n=N_FEATURES_SELECTED, random_state=RANDOM_STATE
)
print("Top selected features:", top_feats[:10])

# Filter datasets to selected features
X_train_sel = X_train[top_feats]
X_val_sel = X_val[top_feats]
X_test_sel = X_test[top_feats]


In [ ]:
# ---------- 5. Model Training ----------
print("Training models ...")

# 1. Random Forest Regressor
print("Training Random Forest Regressor ...")
rf = RandomForestRegressor(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=10,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_sel, y_train_reg)
rf_pred = rf.predict(X_test_sel)

# 2. XGBoost Regressor
print("Training XGBoost Regressor (with early stopping) ...")
xgb_reg = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=40,
    eval_metric="rmse",
)
xgb_reg.fit(
    X_train_sel, y_train_reg,
    eval_set=[(X_val_sel, y_val_reg)],
    verbose=False,
)
xgb_reg_pred = xgb_reg.predict(X_test_sel)
print(f"XGBoost Regressor best iteration: {xgb_reg.best_iteration}")

# 3. XGBoost Classifier (Directional Probability)
print("Training XGBoost Classifier (Directional Probability) ...")
xgb_clf = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=40,
    eval_metric="logloss",
)
xgb_clf.fit(
    X_train_sel, y_train_clf,
    eval_set=[(X_val_sel, y_val_clf)],
    verbose=False,
)
xgb_clf_probs = xgb_clf.predict_proba(X_test_sel)[:, 1]
print(f"XGBoost Classifier best iteration: {xgb_clf.best_iteration}")
print("Training complete.")


In [ ]:
# ---------- 6. Evaluation Metrics ----------
def calc_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "DirAcc": float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        "PctPredNeg": float(np.mean(y_pred < 0)),
    }

rf_metrics = calc_metrics(y_test_reg.values, rf_pred)
xgb_reg_metrics = calc_metrics(y_test_reg.values, xgb_reg_pred)

comparison = pd.DataFrame({"RandomForest": rf_metrics, "XGBoost_Reg": xgb_reg_metrics}).T
comparison_display = comparison.copy()
comparison_display["MAE"] = comparison_display["MAE"].map(lambda x: f"{x:.6f}")
comparison_display["RMSE"] = comparison_display["RMSE"].map(lambda x: f"{x:.6f}")
comparison_display["R2"] = comparison_display["R2"].map(lambda x: f"{x:.5f}")
comparison_display["DirAcc"] = comparison_display["DirAcc"].map(lambda x: f"{x*100:.2f}%")
comparison_display["PctPredNeg"] = comparison_display["PctPredNeg"].map(lambda x: f"{x*100:.2f}%")

print("\n" + "=" * 65)
print("OUT-OF-SAMPLE TEST SET METRICS (Stationary & No Leakage)")
print("=" * 65)
print(comparison_display.to_string())

clf_acc = accuracy_score(y_test_clf.values, (xgb_clf_probs > 0.5).astype(int))
clf_auc = roc_auc_score(y_test_clf.values, xgb_clf_probs)
print(f"\nXGBoost Classifier Directional Accuracy: {clf_acc*100:.2f}%  |  ROC-AUC: {clf_auc:.4f}")
actual_neg_pct = np.mean(y_test_reg.values < 0) * 100
print(f"Actual test market negative returns: {actual_neg_pct:.2f}%")


In [ ]:
# ---------- 7. Trading Strategy Backtest & Simulation ----------
def run_backtest(y_true, signals, spread_cost=SPREAD_PCT):
    pos_changes = np.abs(np.diff(signals, prepend=0))
    gross_returns = signals * y_true
    net_returns = gross_returns - (pos_changes * spread_cost)
    equity_curve = (1.0 + net_returns).cumprod()
    total_return = (equity_curve[-1] - 1.0) * 100
    
    # Annualized Sharpe (hourly: ~6,000 trading hours per year)
    ann_factor = np.sqrt(6000)
    sharpe = (np.mean(net_returns) / (np.std(net_returns) + 1e-9)) * ann_factor
    
    # Max Drawdown
    peak = np.maximum.accumulate(equity_curve)
    drawdowns = (peak - equity_curve) / peak
    max_dd = np.max(drawdowns) * 100
    
    # Hourly win rate (percentage of active bars with positive return)
    active_mask = (signals != 0)
    hourly_win_rate = np.mean(gross_returns[active_mask] > 0) * 100 if np.sum(active_mask) > 0 else 0.0

    # Trade-level statistics (from position entry to exit/flip)
    trades = []
    curr_pos = 0
    trade_ret = 0.0
    for s, r, chg in zip(signals, y_true, pos_changes):
        if chg > 0:
            if curr_pos != 0:
                trades.append(trade_ret)
                trade_ret = 0.0
            curr_pos = s
        if curr_pos != 0:
            trade_ret += (curr_pos * r) - (spread_cost if chg > 0 else 0)
    if curr_pos != 0:
        trades.append(trade_ret)

    trades = np.array(trades)
    trade_win_rate = np.mean(trades > 0) * 100 if len(trades) > 0 else 0.0
    gains = trades[trades > 0].sum() if np.any(trades > 0) else 0.0
    losses = np.abs(trades[trades < 0].sum()) if np.any(trades < 0) else 1e-9
    profit_factor = gains / losses if losses > 0 else np.nan

    return {
        "Total Return (%)": total_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown (%)": max_dd,
        "Hourly Win Rate (%)": hourly_win_rate,
        "Trade Win Rate (%)": trade_win_rate,
        "Profit Factor": profit_factor,
        "Total Trades": int(np.sum(pos_changes > 0)),
        "Equity Curve": equity_curve,
    }

# 1. RF Regressor Signals
sig_rf = np.where(rf_pred > 0, 1, -1)

# 2. XGB Regressor Signals
sig_xgb_reg = np.where(xgb_reg_pred > 0, 1, -1)

# 3. XGB Classifier Conviction Filter (Trade when probability > 0.52 or < 0.48, else stay flat)
sig_xgb_clf = np.zeros(len(xgb_clf_probs))
sig_xgb_clf[xgb_clf_probs > 0.52] = 1
sig_xgb_clf[xgb_clf_probs < 0.48] = -1

y_test_arr = y_test_reg.values
rf_bt = run_backtest(y_test_arr, sig_rf)
xgb_reg_bt = run_backtest(y_test_arr, sig_xgb_reg)
xgb_clf_bt = run_backtest(y_test_arr, sig_xgb_clf)

# Buy & Hold Benchmark
bnh_equity = (1.0 + y_test_arr).cumprod()
bnh_total = (bnh_equity[-1] - 1.0) * 100

bt_summary = pd.DataFrame({
    "RandomForest": {k: v for k, v in rf_bt.items() if k != "Equity Curve"},
    "XGBoost_Reg": {k: v for k, v in xgb_reg_bt.items() if k != "Equity Curve"},
    "XGBoost_Clf_Conviction": {k: v for k, v in xgb_clf_bt.items() if k != "Equity Curve"},
}).T
print("\n" + "=" * 70)
print("TRADING BACKTEST RESULTS (Out-of-sample with transaction costs)")
print("=" * 70)
print(bt_summary.round(2).to_string())
print(f"Buy & Hold Return: {bnh_total:.2f}%")

# Plot Equity Curves
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(y_test_reg.index, rf_bt["Equity Curve"], label=f"Random Forest (Sharpe: {rf_bt['Sharpe Ratio']:.2f})", color="#2980b9", lw=1.5)
ax.plot(y_test_reg.index, xgb_reg_bt["Equity Curve"], label=f"XGBoost Regressor (Sharpe: {xgb_reg_bt['Sharpe Ratio']:.2f})", color="#27ae60", lw=1.5)
ax.plot(y_test_reg.index, xgb_clf_bt["Equity Curve"], label=f"XGBoost Classifier Conviction (Sharpe: {xgb_clf_bt['Sharpe Ratio']:.2f})", color="#8e44ad", lw=1.5)
ax.plot(y_test_reg.index, bnh_equity, label=f"Buy & Hold ({bnh_total:.1f}%)", color="#7f8c8d", linestyle="--", alpha=0.7)

ax.set_title("Out-of-Sample Trading Strategy Equity Curve (XAU/USD Hourly)", fontsize=13, fontweight='bold')
ax.set_ylabel("Portfolio Value (Base = 1.0)")
ax.set_xlabel("Date")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

plt.tight_layout()
curve_path = os.path.join(OUT_DIR, "strategy_equity_curve.png")
plt.savefig(curve_path, dpi=200)
plt.show()
print(f"Saved equity curve plot to {curve_path}")


In [ ]:
# ---------- 8. Export Artifacts ----------
comparison_display.to_csv(os.path.join(OUT_DIR, "metrics_comparison.csv"))
bt_summary.to_csv(os.path.join(OUT_DIR, "backtest_summary.csv"))

rf_imp = pd.Series(rf.feature_importances_, index=top_feats)
xgb_imp = pd.Series(xgb_reg.feature_importances_, index=top_feats)
imp_df = pd.DataFrame({"RF": rf_imp, "XGB": xgb_imp}).sort_values("XGB", ascending=False)
imp_df.to_csv(os.path.join(OUT_DIR, "feature_importances.csv"))

pd.Series(top_feats).to_csv(os.path.join(OUT_DIR, "selected_features.csv"), index=False, header=["feature"])

pred_df = pd.DataFrame(
    {
        "y_true": y_test_reg.values,
        "rf_pred": rf_pred,
        "xgb_reg_pred": xgb_reg_pred,
        "xgb_clf_prob": xgb_clf_probs,
        "close": test_close.values,
    },
    index=y_test_reg.index,
)
pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"))

print(f"\n✅ All artifacts successfully exported to ./{OUT_DIR}/")
print("Done.")
